# 1a단계: ROI 다시 그리기 → 재채점

0단계에서 만든 정답(`gt_roboflow.zip`)과 평가 함수를 그대로 씁니다.

| 순서 | 내용 |
|---|---|
| 공통 설정 · 평가 준비 | 0단계와 같은 셀 (경로, 함수, 정답 불러오기) |
| ①~② | 지금 ROI 확인, 좌표 눈금 이미지 |
| ③ | **새 ROI 좌표 입력** ← 직접 할 일 |
| ④~⑤ | v3 결과에 적용 → 재채점 → Before와 비교 |
| ⑥~⑦ | 잃은 정답 점검, 남은 오류 확인 |

런타임 유형은 **CPU여도 됩니다.**

## 공통 설정

In [ ]:
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

---
# 1a. ROI 다시 그리기

ROI(관심 영역)는 경기장 다각형이에요. 박스의 **발 위치**가 이 안에 있을 때만 선수로 남깁니다.
Before 결과에서 0156의 ROI가 화면 전체로 잡혀 경기장 밖 사람(벤치, 펜스 너머)이 FP로 많이 잡혔어요.

**이번 실험은 탐지를 다시 돌리지 않습니다.** v3 JSON에서 새 ROI 밖의 박스만 지우고 다시 채점해요. (GPU 불필요)

⚠️ **ROI는 경기장 라인을 보고 정하세요. 정답 박스 위치를 보고 맞추면 안 됩니다.**
정답(시험지)에 맞춰 설정을 고르면 점수가 실제보다 좋게 나와요. 같은 이유로, 정답 박스는 ROI를 정한 뒤 확인용으로만 봅니다.

In [ ]:
# ① 지금 v3 ROI 확인: 클립별로 평가 프레임 3장에 v3의 ROI(초록 선)를 그려봄
def frames_of(stem):
    return sorted(n for n in os.listdir(IMG_DIR) if n.startswith(stem) and n.endswith(".jpg"))

def show_roi(stem, poly, title, n=3, color=(0, 255, 0)):
    files = frames_of(stem)
    picks = files[:: max(1, len(files) // n)][:n]
    fig, axes = plt.subplots(1, len(picks), figsize=(7 * len(picks), 4.5))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, picks):
        img = cv2.imread(os.path.join(IMG_DIR, f))
        if poly is not None and len(poly) >= 3:
            cv2.polylines(img, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], True, color, 3)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f, fontsize=9); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

V3_ROI = {}
for stem in EVAL_CLIPS:
    _, poly = load_tracking(tracking_json_of(stem))
    V3_ROI[stem] = poly
    H, W = cv2.imread(os.path.join(IMG_DIR, frames_of(stem)[0])).shape[:2]
    print(f"{stem}: 프레임 {W}x{H}, v3 ROI 꼭짓점 {len(poly)}개 → {poly.tolist()}")
    show_roi(stem, poly, f"{stem} — v3 ROI")

In [ ]:
# ② 좌표 읽기용 눈금 이미지: 100px마다 선, 200px마다 숫자
#    경기장(사이드라인·엔드라인) 꼭짓점의 (x, y)를 읽어서 다음 셀에 적으세요.
def show_grid(stem, idx=0, step=100):
    f = frames_of(stem)[idx]
    img = cv2.imread(os.path.join(IMG_DIR, f))
    H, W = img.shape[:2]
    for x in range(0, W, step):
        cv2.line(img, (x, 0), (x, H), (255, 255, 255), 1)
        if x % (2 * step) == 0:
            cv2.putText(img, str(x), (x + 3, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    for y in range(0, H, step):
        cv2.line(img, (0, y), (W, y), (255, 255, 255), 1)
        if y % (2 * step) == 0:
            cv2.putText(img, str(y), (5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    plt.figure(figsize=(20, 11.5)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{f}  (노란 숫자 = 픽셀 좌표)"); plt.axis("off"); plt.show()

for stem in EVAL_CLIPS:
    show_grid(stem)

In [ ]:
# ③ 새 ROI 입력
#    - 경기장 라인을 따라 꼭짓점을 시계방향(또는 반시계방향)으로 적어요. 4~6개면 충분합니다.
#    - None으로 두면 v3 ROI를 그대로 씁니다 (문제가 없는 클립은 None).
#    - 화면 밖으로 이어지는 라인은 화면 끝 좌표(x=0, x=W-1, y=H-1 등)를 쓰면 돼요.
NEW_ROI = {
    "DJI_20250913133612_0144_D": None,
    "DJI_20250913135020_0156_D": [[0, 0], [0, 0], [0, 0], [0, 0]],   # ← 직접 채우기
}

# 사이드라인에 붙어 선 선수가 잘리지 않도록 라인보다 바깥으로 넓히는 여유(px)
ROI_MARGIN_PX = 20


def expand_polygon(poly, margin, W, H):
    """다각형을 margin(px)만큼 바깥으로 넓힘: 채운 마스크를 팽창(dilate)시킨 뒤 다시 다각형으로."""
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], 255)
    if margin > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1))
        mask = cv2.dilate(mask, k)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    c = cv2.approxPolyDP(c, 2.0, True)
    return c.reshape(-1, 2)


FINAL_ROI = {}
for stem, pts in NEW_ROI.items():
    if pts is None:
        FINAL_ROI[stem] = V3_ROI[stem]
        print(f"{stem}: v3 ROI 유지")
        continue
    pts = np.asarray(pts, np.int32)
    assert len(pts) >= 3 and np.ptp(pts[:, 0]) > 0 and np.ptp(pts[:, 1]) > 0, \
        f"{stem}: NEW_ROI 좌표를 채워주세요 (지금은 비어 있거나 한 점에 몰려 있어요)"
    H, W = cv2.imread(os.path.join(IMG_DIR, frames_of(stem)[0])).shape[:2]
    FINAL_ROI[stem] = expand_polygon(pts, ROI_MARGIN_PX, W, H)
    print(f"{stem}: 새 ROI (여유 {ROI_MARGIN_PX}px 적용) 꼭짓점 {len(FINAL_ROI[stem])}개")

# 확인: 노랑 = 입력한 라인, 초록 = 여유를 준 최종 ROI. 여러 프레임에서 경기장을 잘 감싸는지 보세요.
for stem in EVAL_CLIPS:
    if NEW_ROI[stem] is None:
        continue
    files = frames_of(stem)
    picks = files[:: max(1, len(files) // 3)][:3]
    fig, axes = plt.subplots(1, len(picks), figsize=(7 * len(picks), 4.5))
    for ax, f in zip(np.atleast_1d(axes), picks):
        img = cv2.imread(os.path.join(IMG_DIR, f))
        cv2.polylines(img, [np.asarray(NEW_ROI[stem], np.int32).reshape(-1, 1, 2)], True, (0, 255, 255), 2)
        cv2.polylines(img, [FINAL_ROI[stem].reshape(-1, 1, 2)], True, (0, 255, 0), 3)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f, fontsize=9); ax.axis("off")
    fig.suptitle(f"{stem} — 노랑: 입력 / 초록: 최종 ROI"); plt.tight_layout(); plt.show()

In [ ]:
# ④ v3 JSON에 새 ROI 적용 → 새 JSON 저장 (탐지는 다시 돌리지 않음)
RUN_NAME = "step1a_roi"
RUN_DIR = os.path.join(EVAL_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)


def apply_roi(stem, poly):
    with open(tracking_json_of(stem), encoding="utf-8") as f:
        data = json.load(f)
    before = after = 0
    for fr in data["frames"]:
        before += len(fr["players"])
        fr["players"] = [p for p in fr["players"] if inside(poly, *p["feet"])]
        after += len(fr["players"])
    data["field_polygon"] = np.asarray(poly).tolist()
    data.setdefault("notes", []).append(f"{RUN_NAME}: v3 결과에 새 ROI를 후처리로 적용 (여유 {ROI_MARGIN_PX}px)")
    out = os.path.join(RUN_DIR, f"{stem}_tracking_final.json")
    with open(out, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"{stem}: 전체 박스 {before:,} → {after:,} (제거 {before - after:,})")
    return out


STEP1A_RUN = {stem: apply_roi(stem, FINAL_ROI[stem]) for stem in EVAL_CLIPS}

In [ ]:
# ⑤ 다시 채점 + Before와 비교
row_1a, conf_1a, pf_1a = evaluate_run(RUN_NAME, STEP1A_RUN, note=f"v3 + ROI 재설정(여유 {ROI_MARGIN_PX}px), 재탐지 없음")

COMPARE_COLS = ["run", "det_precision", "det_recall", "det_f1", "TP", "FP", "FN", "team_acc", "referee_recall",
                "segment_unique_ids_34", "segment_unique_ids_44", "segment_unknown_ratio_34", "segment_unknown_ratio_44"]
res = pd.read_csv(RESULTS_CSV)
cmp = res[res["run"].isin(["v3_baseline", RUN_NAME])][COMPARE_COLS].set_index("run").T
if {"v3_baseline", RUN_NAME} <= set(cmp.columns):
    cmp["변화"] = cmp[RUN_NAME] - cmp["v3_baseline"]
display(cmp)

In [ ]:
# ⑥ 안전 점검: ROI 때문에 '잃어버린 정답'이 있는지
#    Before에서는 맞게 찾았는데(TP) 새 ROI가 지워버린 선수 = ROI가 너무 좁다는 신호
lost = []
for stem in EVAL_CLIPS:
    v3_by, _ = load_tracking(tracking_json_of(stem))
    new_by, _ = load_tracking(STEP1A_RUN[stem])
    for name, g in GT.items():
        if not name.startswith(stem):
            continue
        fid = int(name.split("_f")[-1])
        gb = [b for _, b in g["boxes"]]
        m_old = {gi for gi, _ in match_by_center(gb, [p["bbox"] for p in v3_by.get(fid, [])])[0]}
        m_new = {gi for gi, _ in match_by_center(gb, [p["bbox"] for p in new_by.get(fid, [])])[0]}
        for gi in sorted(m_old - m_new):
            lost.append((name, g["boxes"][gi][0], [round(v) for v in g["boxes"][gi][1]]))

print(f"ROI 때문에 잃은 정답: {len(lost)}개")
for x in lost[:20]:
    print("  ", x)
if lost:
    print("→ ROI_MARGIN_PX를 늘리거나 꼭짓점을 바깥으로 옮긴 뒤 ③부터 다시 실행하세요.")
else:
    print("✅ 잃은 정답 없음: 줄어든 FP는 전부 경기장 밖 오탐이었어요.")

In [ ]:
# ⑦ 남은 오류 확인: FP가 가장 많은 프레임 2장 (파랑 = 아직 남은 오탐)
for name in pf_1a.sort_values("FP", ascending=False)["image"].head(2):
    show_errors(name, STEP1A_RUN)

## 결과 기록
`eval_results.csv`에 `step1a_roi` 줄이 추가됐어요. 비교표를 캡처해서 공유해주세요.

**해석할 때 주의:** 이번 실험은 v3 결과에서 박스를 지우기만 했어요. v3의 '팀당 11명' 규칙은 예전 ROI 기준으로 계산된 상태라,
파이프라인을 새 ROI로 처음부터 다시 돌리면 팀 분류와 Unknown 비율이 조금 달라질 수 있어요. (탐지 Precision/Recall은 그대로)